In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import numpy as np

# Set up device
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Tokenizer
tokenizer = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50')

# Preprocess verb data without morphological features
def preprocess_baseline_data(verbs_data):
    processed_data = []
    for item in verbs_data:
        sentence = item['sentence']
        translation = item['translation']
        verb_infos = item.get('verb_info', [])  # Get verb_info, default to empty list if not present
        if not isinstance(verb_infos, list):
            verb_infos = [verb_infos]  # Ensure verb_infos is always a list
        
        processed_data.append({
            'original_sentence': sentence,
            'processed_sentence': sentence,  # No morphological tags in baseline
            'translation': translation,
            'verb_infos': verb_infos  # Include verb_infos in the processed data
        })
    return processed_data

# Split data
def split_data(processed_data, test_size=0.1, val_size=0.1):
    train_val, test = train_test_split(processed_data, test_size=test_size, random_state=42)
    train, val = train_test_split(train_val, test_size=val_size/(1-test_size), random_state=42)
    return train, val, test

# Dataset class
class ArmenianVerbDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.morpho_features = ['tense', 'aspect', 'mood', 'voice', 'person', 'number']
        self.feature_to_id = {
            'tense': {'past': 0, 'present': 1, 'future': 2, 'unknown': 3},
            'aspect': {'perfective': 0, 'imperfective': 1, 'unknown': 2},
            'mood': {'indicative': 0, 'subjunctive': 1, 'imperative': 2, 'unknown': 3},
            'voice': {'active': 0, 'passive': 1, 'unknown': 2},
            'person': {'1': 0, '2': 1, '3': 2, 'unknown': 3},
            'number': {'singular': 0, 'plural': 1, 'unknown': 2}
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        input_text = item['processed_sentence']
        target_text = item['original_sentence']
        
        inputs = self.tokenizer(input_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        targets = self.tokenizer(target_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')

        input_ids = inputs.input_ids.squeeze()
        attention_mask = inputs.attention_mask.squeeze()
        labels = targets.input_ids.squeeze()

        # Add morphological features
        morpho_labels = {}
        if item['verb_infos']:
            for feature in self.morpho_features:
                value = item['verb_infos'][0].get('analysis', {}).get(feature, 'unknown')
                morpho_labels[f"{feature}_labels"] = torch.tensor(self.feature_to_id[feature].get(value, self.feature_to_id[feature]['unknown']))
        else:
            morpho_labels = {f"{feature}_labels": torch.tensor(self.feature_to_id[feature]['unknown']) for feature in self.morpho_features}

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            **morpho_labels
        }

# Model Training for Baseline
def train_baseline(model, train_dataloader, val_dataloader, optimizer, scheduler, tokenizer, device, num_epochs=10, save_dir='./baseline_model_checkpoints'):
    model.to(device)
    best_val_loss = float('inf')
    training_history = []
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            optimizer.zero_grad()
            
            inputs = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'labels']}
            
            outputs = model(**inputs)
            loss = outputs.loss
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_dataloader)
        
        # Validation
        val_loss, bleu_score, morpho_accuracies = evaluate_baseline(model, val_dataloader, tokenizer, device)
        
        epoch_info = {
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_loss': val_loss,
            'bleu_score': bleu_score,
            'morpho_accuracies': morpho_accuracies
        }
        training_history.append(epoch_info)
        
        print(f"Epoch {epoch+1}")
        print(f"  Training Loss: {avg_train_loss:.4f}")
        print(f"  Validation Loss: {val_loss:.4f}")
        print(f"  Validation BLEU Score: {bleu_score:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model_path = os.path.join(save_dir, f'best_model_epoch_{epoch+1}')
            model.save_pretrained(model_path)
            print(f"  Best model saved to {model_path}")

    return model, training_history

# Baseline evaluation function (BLEU score only)
def evaluate_baseline(model, dataloader, tokenizer, device):
    model.eval()
    total_loss = 0
    all_translations = []
    all_references = []
    all_morpho_true = {feature: [] for feature in ['tense', 'aspect', 'mood', 'voice', 'person', 'number']}
    all_morpho_preds = {feature: [] for feature in ['tense', 'aspect', 'mood', 'voice', 'person', 'number']}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating Baseline"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'labels']}
            
            outputs = model(**inputs)
            loss = outputs.loss
            total_loss += loss.item()
            
            # Generate translations
            generated = model.generate(inputs['input_ids'], attention_mask=inputs['attention_mask'])
            
            translations = tokenizer.batch_decode(generated, skip_special_tokens=True)
            references = tokenizer.batch_decode(inputs['labels'], skip_special_tokens=True)
            
            all_translations.extend(translations)
            all_references.extend(references)
            
            # Collect true morphological features
            for feature in all_morpho_true.keys():
                all_morpho_true[feature].extend(batch[f"{feature}_labels"].cpu().numpy())
                
            # For morphological predictions, we'll use a simple heuristic
            # For example, we could check if certain words are present in the translation
            for trans in translations:
                preds = {
                    'tense': 1 if 'will' in trans else 0,  # Simple future tense detection
                    'aspect': 1 if 'ing' in trans else 0,  # Simple progressive aspect detection
                    'mood': 1 if '?' in trans else 0,  # Simple interrogative mood detection
                    'voice': 1 if 'by' in trans else 0,  # Simple passive voice detection
                    'person': 1 if 'I' in trans else (2 if 'you' in trans else 0),  # Simple person detection
                    'number': 1 if 'they' in trans or 'we' in trans else 0  # Simple plural detection
                }
                for feature in all_morpho_preds.keys():
                    all_morpho_preds[feature].append(preds[feature])

    avg_loss = total_loss / len(dataloader)
    bleu_score = corpus_bleu([[ref.split()] for ref in all_references], [hyp.split() for hyp in all_translations])
    
    morpho_accuracies = {}
    for feature in all_morpho_preds.keys():
        morpho_accuracies[feature] = accuracy_score(all_morpho_true[feature], all_morpho_preds[feature])

    return avg_loss, bleu_score, morpho_accuracies

# Load Data (you need to load your actual Armenian verbs data here)
from armenian_verbs import verbs_data

# Preprocess the data for baseline
processed_data = preprocess_baseline_data(verbs_data)
train_data, val_data, test_data = split_data(processed_data)

# Create Datasets and Dataloaders
train_dataset = ArmenianVerbDataset(train_data, tokenizer)
val_dataset = ArmenianVerbDataset(val_data, tokenizer)
test_dataset = ArmenianVerbDataset(test_data, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)
test_dataloader = DataLoader(test_dataset, batch_size=16)


# Initialize the baseline model
baseline_model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50")

# Set optimizer and scheduler
optimizer = AdamW(baseline_model.parameters(), lr=5e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=500, num_training_steps=len(train_dataloader) * 10)

# Train the baseline model
trained_baseline_model, training_history = train_baseline(baseline_model, train_dataloader, val_dataloader, optimizer, scheduler, tokenizer, device)

# Save the final model
final_model_path = './baseline_model_checkpoints/final_model'
trained_baseline_model.save_pretrained(final_model_path)
print(f"Final baseline model saved to {final_model_path}")




In [ ]:
def evaluate_baseline(model, dataloader, tokenizer, device):
    model.eval()
    total_loss = 0
    all_inputs = []
    all_references = []
    all_translations = []
    all_morpho_true = {feature: [] for feature in ['tense', 'aspect', 'mood', 'voice', 'person', 'number']}
    all_morpho_preds = {feature: [] for feature in ['tense', 'aspect', 'mood', 'voice', 'person', 'number']}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating Baseline"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'labels']}
            
            outputs = model(**inputs)
            loss = outputs.loss
            total_loss += loss.item()
            
            # Generate translations
            generated = model.generate(inputs['input_ids'], attention_mask=inputs['attention_mask'])
            
            translations = tokenizer.batch_decode(generated, skip_special_tokens=True)
            references = tokenizer.batch_decode(inputs['labels'], skip_special_tokens=True)
            input_texts = tokenizer.batch_decode(inputs['input_ids'], skip_special_tokens=True)
            
            all_inputs.extend(input_texts)
            all_translations.extend(translations)
            all_references.extend(references)
            
            # Collect true morphological features
            for feature in all_morpho_true.keys():
                all_morpho_true[feature].extend(batch[f"{feature}_labels"].cpu().numpy())
            
            # For baseline, we'll use a simple heuristic for predictions
            for trans in translations:
                preds = {
                    'tense': 0,  # Default prediction
                    'aspect': 0,
                    'mood': 0,
                    'voice': 0,
                    'person': 0,
                    'number': 0
                }
                for feature in all_morpho_preds.keys():
                    all_morpho_preds[feature].append(preds[feature])

    avg_loss = total_loss / len(dataloader)
    bleu_score = corpus_bleu([[ref.split()] for ref in all_references], [hyp.split() for hyp in all_translations])
    
    morpho_accuracies = {}
    for feature in all_morpho_preds.keys():
        morpho_accuracies[feature] = accuracy_score(all_morpho_true[feature], all_morpho_preds[feature])

    return avg_loss, bleu_score, morpho_accuracies, all_inputs, all_references, all_translations, all_morpho_true, all_morpho_preds
    
# Import inference data
from armenian_verbs_inference import inference_verbs_data

# Preprocess the inference data
processed_inference_data = preprocess_baseline_data(inference_verbs_data)

# Create inference dataset
inference_dataset = ArmenianVerbDataset(processed_inference_data, tokenizer)

# Create inference dataloader
inference_dataloader = DataLoader(inference_dataset, batch_size=16, shuffle=False)

# Evaluate the baseline on the test set
print("\n--- Evaluating Baseline on Test Data ---\n")

# Evaluate the baseline on the test set
print("\n--- Evaluating Baseline on Test Data ---\n")
baseline_test_results = evaluate_baseline(trained_baseline_model, test_dataloader, tokenizer, device)
baseline_test_loss, baseline_bleu_test, baseline_morpho_accuracies_test = baseline_test_results[:3]
test_inputs, test_references, test_translations, test_morpho_true, test_morpho_preds = baseline_test_results[3:]

print(f"Baseline Test Loss: {baseline_test_loss:.4f}")
print(f"Baseline Test BLEU Score: {baseline_bleu_test:.4f}")
print("Baseline Test Morphological Accuracies:")
for feature, accuracy in baseline_morpho_accuracies_test.items():
    print(f"  {feature.capitalize()}: {accuracy:.4f}")

# Print detailed results for test data
print("\nDetailed Test Results:")
for i in range(min(10, len(test_inputs))):  # Print first 10 instances
    print(f"\nInstance {i+1}:")
    print(f"Input: {test_inputs[i]}")
    print(f"Reference: {test_references[i]}")
    print(f"Model Output: {test_translations[i]}")
    print("Morphological features:")
    for feature in test_morpho_true.keys():
        print(f"  {feature.capitalize()}: True - {test_morpho_true[feature][i]}, Predicted - {test_morpho_preds[feature][i]}")
    print("-" * 50)

# Evaluate the baseline on inference data
print("\n--- Evaluating Baseline on Inference Data ---\n")
baseline_inf_results = evaluate_baseline(trained_baseline_model, inference_dataloader, tokenizer, device)
baseline_inf_loss, baseline_bleu_inf, baseline_morpho_accuracies_inf = baseline_inf_results[:3]
inf_inputs, inf_references, inf_translations, inf_morpho_true, inf_morpho_preds = baseline_inf_results[3:]

print(f"Baseline Inference Loss: {baseline_inf_loss:.4f}")
print(f"Baseline Inference BLEU Score: {baseline_bleu_inf:.4f}")
print("Baseline Inference Morphological Accuracies:")
for feature, accuracy in baseline_morpho_accuracies_inf.items():
    print(f"  {feature.capitalize()}: {accuracy:.4f}")

# Print detailed results for inference data
print("\nDetailed Inference Results:")
for i in range(len(inf_inputs)):
    print(f"\nInstance {i+1}:")
    print(f"Input: {inf_inputs[i]}")
    print(f"Reference: {inf_references[i]}")
    print(f"Model Output: {inf_translations[i]}")
    print("Morphological features:")
    for feature in inf_morpho_true.keys():
        print(f"  {feature.capitalize()}: True - {inf_morpho_true[feature][i]}, Predicted - {inf_morpho_preds[feature][i]}")
    print("-" * 50)